In [1]:
import pandas as pd
import re
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

df = pd.read_csv("test.csv")

# 所在地の整形
def format_address(address):
    # 括弧内の文字と「未定」を削除
    address = re.sub(r'[（(].*?[）)]|未定|詳細|番|号', '', address)
    # 番地や後ろの数字、ハイフン（全角・半角）を削除
    #address = re.sub(r'([－-]?\d+|[-－]\d*|[－-])$', '', address)  # 全角・半角のハイフンと数字を削除
    address = re.sub(r'([ー－-]?\d+|[-－ー]\d*)$', '', address)  # 全角・半角のハイフンと数字を削除
    address = re.sub(r'([ー－-]?\d+|[-－ー]\d*)$', '', address)  # ハイフンと数字の組み合わせを削除
    return address.strip()

df['整形住所'] = df['所在地'].apply(format_address)

# Geolocatorのインスタンスを作成
geolocator = Nominatim(user_agent="your_app_name")

def get_lat_long(location):
    retries = 3  # 最大リトライ回数
    for attempt in range(retries):
        try:
            location = geolocator.geocode(location, timeout=10)  # タイムアウトを10秒に設定
            if location:
                return location.latitude, location.longitude
            return None, None
        except GeocoderTimedOut:
            if attempt < retries - 1:  # 最後の試行でない場合
                time.sleep(2)  # 2秒待機してリトライ
            else:
                print(f"Geocoder service timed out for {location}.")
                return None, None

# データを分割する
chunk_size = 10000  # 分割サイズ
latitudes = []
longitudes = []

for i in range(0, len(df), chunk_size):
    chunk = df.iloc[i:i + chunk_size]
    for loc in chunk['整形住所']:
        lat, lon = get_lat_long(loc)
        latitudes.append(lat)
        longitudes.append(lon)
        time.sleep(2)  # リクエストのスリープ（2秒）

# 経度と緯度を取得
df['緯度'] = latitudes
df['経度'] = longitudes

# 結果を表示
print(df[['id', '整形住所', '緯度', '経度']])


          id           整形住所         緯度          経度
0      31471   東京都世田谷区深沢５丁目  35.618736  139.649580
1      31472    東京都目黒区八雲１丁目        NaN         NaN
2      31473  東京都豊島区池袋本町２丁目  35.741658  139.710592
3      31474    東京都杉並区和泉１丁目  35.675235  139.657377
4      31475   東京都杉並区堀ノ内２丁目  35.688220  139.652600
...      ...            ...        ...         ...
31257  62728   東京都豊島区上池袋４丁目  35.740990  139.720947
31258  62729  東京都千代田区岩本町２丁目  35.693526  139.776933
31259  62730   東京都中野区江古田３丁目  35.728498  139.666102
31260  62731      東京都千代田区二町        NaN         NaN
31261  62732   東京都大田区南馬込６丁目  35.583385  139.708457

[31262 rows x 4 columns]


In [2]:
df.to_csv('df_test.csv', index=False)